# 📄 FILE: 04_inference.py

## 🎯 Chức năng chính
Đây là bước cuối cùng và thú vị nhất: **Kiểm thử thực tế (Inference)**.
Script này sẽ đóng vai trò là giao diện "Trợ lý ảo", cho phép bạn nhập triệu chứng bệnh vào và nhận lại kết quả chẩn đoán từ mô hình AI đã huấn luyện.

## ⚙️ Quy trình xử lý
1.  **Load Architecture:** Tự động import cấu trúc mô hình (`DiabetesHybridModel`) từ file `02_model_architecture.py`.
2.  **Load Weights:** Tải bộ trọng số thông minh nhất (`best_model.pth`) mà bạn vừa train được ở bước trước.
3.  **Predict Loop:**
    * Nhận văn bản input từ bàn phím.
    * Xử lý qua Tokenizer (ViBERT) và tính điểm Từ điển.
    * Đưa vào mô hình để tính toán xác suất (%).
    * In kết quả và lời khuyên ra màn hình.

In [2]:
# ==============================================================================
# FILE: 04_inference.py
# CHỨC NĂNG: Chat với mô hình AI để chẩn đoán bệnh
# ==============================================================================

In [1]:
import torch
from transformers import AutoTokenizer
import importlib
import sys
import os

In [3]:
# --- BƯỚC 1: IMPORT DYNAMIC TỪ FILE 02 ---
try:
    arch = importlib.import_module("02_model_architecture")
    DiabetesDataset = arch.DiabetesDataset
    DiabetesHybridModel = arch.DiabetesHybridModel
    MODEL_NAME = arch.MODEL_NAME
    MAX_LEN = arch.MAX_LEN
    print("Đã import cấu trúc mô hình từ File 02.")
except ImportError:
    print("❌ Lỗi: Không tìm thấy file '02_model_architecture.py'.")
    print("👉 Hãy chắc chắn bạn đang chạy file này trong cùng thư mục dự án.")
    # exit() # Nếu chạy trong Notebook thì comment dòng này lại

Đã import cấu trúc mô hình từ File 02.


In [4]:
# --- CẤU HÌNH THIẾT BỊ ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ Đang chạy trên thiết bị: {DEVICE}")

⚙️ Đang chạy trên thiết bị: cuda


In [ ]:
# --- BƯỚC 2: LOAD MODEL & TOKENIZER ---
print("⏳ Đang khởi tạo mô hình...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = DiabetesHybridModel(n_classes=2)

model_path = 'best_model.pth'

if os.path.exists(model_path):
    # Load trọng số đã train vào mô hình
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval() # Chuyển sang chế độ dự đoán (tắt Dropout)
    print(f"✅ Đã nạp thành công trọng số từ '{model_path}'!")
else:
    print(f"❌ Lỗi: Không tìm thấy file '{model_path}'.")
    print("👉 Bạn cần chạy File 03 để train mô hình trước đã!")
    # exit()

⏳ Đang khởi tạo mô hình...


config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/581M [00:00<?, ?B/s]

In [ ]:
# --- BƯỚC 3: HÀM DỰ ĐOÁN ---
# Tạo dataset giả để dùng lại hàm tính điểm từ điển (extract_weighted_features)
# Lưu ý: Phải có file 'diabetes_keywords.json' trong thư mục
try:
    helper_ds = DiabetesDataset([], [], [], 'diabetes_keywords.json', tokenizer)
except:
    print("⚠️ Cảnh báo: Không tìm thấy 'diabetes_keywords.json'. Điểm từ điển sẽ bằng 0.")
    helper_ds = DiabetesDataset([], [], [], '', tokenizer)

def predict_symptom(text):
    """Hàm nhận text -> Trả về nhãn dự đoán và độ tin cậy"""
    
    # 1. Xử lý văn bản cho ViBERT
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    
    # 2. Tính điểm từ điển
    dict_feat = helper_ds.extract_weighted_features(text)
    
    # 3. Đưa dữ liệu lên GPU/CPU
    input_ids = encoding['input_ids'].flatten().unsqueeze(0).to(DEVICE)
    attention_mask = encoding['attention_mask'].flatten().unsqueeze(0).to(DEVICE)
    dict_features = torch.tensor([dict_feat], dtype=torch.float).to(DEVICE)

    # 4. Mô hình suy luận
    with torch.no_grad(): # Không tính gradient để tiết kiệm nhớ
        outputs = model(input_ids, attention_mask, dict_features)
        
        # Tính xác suất (Softmax)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        confidence, prediction = torch.max(probs, dim=1)

    return prediction.item(), confidence.item()